# RTM-Only Introduction: Generating a Signal in Each Area

This notebook focuses on RTM alone. It creates one RTM cell for each area in the connectivity data and examines its membrane voltage and spikes.

The goal is to understand how RTM input produces an electrical signal.


## Step 1: Import Tools and Locate the Project

Locate the project root and import Brian2 and the tools for reading area data. If a module cannot be found, make sure Jupyter is using the project's .venv environment.


In [ ]:
from pathlib import Path
import sys
project_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "data").is_dir() and (p / "template_kwargs.py").is_file()),
    None,
)
if project_root is None:
    raise RuntimeError("Start Jupyter from the project root or one of its subdirectories.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import matplotlib.pyplot as plt
from brian2 import ms
from template_kwargs import template_kwargs
from model_class import RTMAreaModel
print("Project root:", project_root)


## Step 2: Read the Area List

The area_list in the connectivity data is used only to get area names and the number of areas. FLN is not used to create connections between areas here.


In [ ]:
connectivity_path = project_root / "data" / template_kwargs["connectivity_file"]
with open(connectivity_path, "rb") as f:
    anatomy = np.load(f, allow_pickle=True, encoding="bytes")
area_names = anatomy[b"area_list"]
area_labels = [
    x.decode("utf-8") if isinstance(x, bytes) else str(x) for x in area_names
]
n_areas = len(area_labels)
print("Number of areas:", n_areas)
print("First few areas:", area_labels[:10])


## Step 3: Set the Simulation Duration and Input

These parameters come from RTMmodel.m: baseline current is 0, pulse amplitude is 0.12, and the pulse starts at 100 ms and ends at 5000 ms. This notebook runs for only 500 ms, so the pulse stays on until the simulation ends.

The MATLAB file does not specify current units. Here, 0.12 is provisionally interpreted as µA/cm², a common convention.


In [ ]:
run_duration = 500 * ms
integration_dt = 0.01 * ms
base_current = 0.0
pulse_current = 0.24
pulse_on = 100 * ms
pulse_off = 5000 * ms
print("Run duration:", run_duration / ms, "ms")
print("Integration time step:", integration_dt / ms, "ms")
print("Pulse starts at:", pulse_on / ms, "ms")


## Step 4: RTM Dynamics

The RTM equations are implemented in RTMAreaModel in model_class.py. They follow the voltage and gating equations from RTMmodel.m. m is set to its steady-state value at the current voltage; h and n change over time. The exprel function keeps rate calculations numerically stable when a denominator approaches zero.


In [ ]:
print(RTMAreaModel.equations)


## Step 5: Create One RTM Cell for Each Area

RTMAreaModel creates one RTM cell for each area, with no inter-area synapses. V4 receives the current pulse and all other areas remain at baseline current. This area-specific input is a teaching extension; RTMmodel.m describes only one cell.


In [ ]:
rtm_model = RTMAreaModel(
    area_labels,
    run_duration=run_duration,
    integration_dt=integration_dt,
    base_current=base_current,
    pulse_current=pulse_current,
    pulse_on=pulse_on,
    pulse_off=pulse_off,
    stimulated_areas=["V4"],
)
v4_index = rtm_model.area_names.index("V4")
print("RTM cells created:", len(rtm_model.area_names))
print("Stimulated area:", rtm_model.area_names[v4_index])
print("All other areas receive baseline current.")


## Step 6: Run RTM

This cell runs the RTM cells. V4 receives the pulse and all other areas receive baseline current.


In [ ]:
rtm_model.run()
print("RTM simulation complete.")


## Step 7: Inspect the Voltage Results

Convert voltage to mV. Rows in the array are time points and columns are areas. Check the data shape and range before plotting.


In [ ]:
time_ms = rtm_model.time_ms
voltage_mv = rtm_model.voltage_mv
spike_times_by_area = rtm_model.spike_times_by_area

print("Array shape (time points, areas):", voltage_mv.shape)
print("Minimum voltage:", np.nanmin(voltage_mv), "mV")
print("Maximum voltage:", np.nanmax(voltage_mv), "mV")


## Step 8: Plot the Input and Membrane Voltage

The top plot shows the pulse delivered to V4; the bottom plot shows V4's membrane-voltage response. Membrane voltage is the direct simulated RTM signal.


In [ ]:
input_trace = rtm_model.input_current_uA_cm2[:, v4_index]
fig, axes = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
axes[0].plot(time_ms, input_trace)
axes[0].set_ylabel("Input current (µA/cm²)")
axes[0].set_title("Current delivered to V4")
axes[1].plot(time_ms, voltage_mv[:, v4_index])
axes[1].set_ylabel("Membrane voltage (mV)")
axes[1].set_xlabel("Time (ms)")
axes[1].set_title("RTM voltage: V4")
for ax in axes:
    ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


## Step 9: Compare RTM Outputs Across Areas

V4 has a different input, so its trace can differ. The other areas share identical initial states, parameters, and baseline input, so their traces should overlap. No activity propagates between areas because there are no inter-area connections.


In [ ]:
n_show = min(5, n_areas)
plt.figure(figsize=(9, 4))
indices_to_show = [area_labels.index("V4")] + [
    idx for idx, label in enumerate(area_labels) if label != "V4"
][:n_show - 1]
for idx in indices_to_show:
    plt.plot(time_ms, voltage_mv[:, idx], label=area_labels[idx], alpha=0.8)
plt.xlabel("Time (ms)")
plt.ylabel("Membrane voltage (mV)")
plt.title("RTM signals across areas")
plt.legend()
plt.grid(alpha=0.25)
plt.show()
print("Maximum voltage difference among the displayed areas:",
      np.max(np.ptp(voltage_mv[:, indices_to_show], axis=1)), "mV")


## Step 10: View the Detected Spikes

RTMAreaModel detects local voltage maxima above -10 mV, matching the peak-height criterion in RTMmodel.m. This plot shows the RTM spike times in each area.


In [ ]:
plt.figure(figsize=(9, 4))
for idx, spike_times in enumerate(spike_times_by_area):
    plt.scatter(spike_times, np.full(len(spike_times), idx), s=12, color="black")
plt.xlabel("Time (ms)")
plt.ylabel("Area index")
plt.title("RTM spike times")
plt.grid(axis="x", alpha=0.25)
plt.show()
print("Spike counts in the first few areas:",
      [len(x) for x in spike_times_by_area[:10]])


## Scope of This Model

- Each area has one RTM cell. All cells currently have the same initial state and input, so their outputs overlap.
- There are no inter-area connections or large-scale network dynamics in this walkthrough.
- To make areas produce different outputs, assign them different inputs or initial states in a later step.
- Brian2 uses fixed-step RK4, while the MATLAB source uses ode15s, so numerical traces may differ.
- The MATLAB file does not specify current units; this notebook provisionally assumes µA/cm².
